# 🐍 Day 2: Pydantic Models

- Pydantic validation
- Request and response models
- Field validators and constraints
- Settings management

## Basic Pydantic Model

Define classes with type hints. Pydantic validates and coerces input.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    id: int
    name: str
    email: str
    active: bool = True

user = User(id=1, name="Alice", email="a@b.com")
print(user.model_dump())
print(user.model_dump_json())

## Field Validation

Use `Field()` for constraints: min_length, max_length, regex, gt, lt, etc.

In [ ]:
from pydantic import Field

class Product(BaseModel):
    name: str = Field(min_length=1, max_length=100)
    price: float = Field(gt=0, le=10000)
    sku: str = Field(pattern=r"^[A-Z]{3}-[0-9]{4}$")

p = Product(name="Widget", price=9.99, sku="ABC-1234")
print(p)
# Product(name='Widget', price=9.99, sku='ABC-1234')

## Optional and Default Values

Use `Optional[T]` or `T = default` for optional fields.

In [ ]:
from typing import Optional

class Item(BaseModel):
    name: str
    description: Optional[str] = None
    quantity: int = 0

i1 = Item(name="X")
i2 = Item(name="Y", description="Desc", quantity=5)
print(i1, i2)

## Nested Models

Use other Pydantic models as field types.

In [ ]:
class Address(BaseModel):
    street: str
    city: str
    zip_code: str

class Person(BaseModel):
    name: str
    address: Address

p = Person(name="Alice", address={"street": "123 Main", "city": "NYC", "zip_code": "10001"})
print(p.address.city)

## model_dump and model_dump_json

`model_dump()` -> dict, `model_dump_json()` -> JSON string. Use `exclude_unset=True` to omit defaults.

In [ ]:
user = User(id=1, name="Bob", email="b@b.com", active=False)
print("Full:", user.model_dump())
print("Exclude unset:", user.model_dump(exclude_unset=True))
print("Exclude id:", user.model_dump(exclude={"id"}))

## Request vs Response Models

Create separate models: UserCreate (no id), UserOut (no password), UserInDB (full).

In [ ]:
class UserCreate(BaseModel):
    username: str
    email: str
    password: str

class UserOut(BaseModel):
    id: int
    username: str
    email: str
    # No password!

print("Separate models for create (input) vs response (output)")

## Settings with pydantic-settings

Load config from env vars. `pip install pydantic-settings`

In [ ]:
from pydantic_settings import BaseSettings

class Settings(BaseSettings):
    app_name: str = "My API"
    debug: bool = False
    database_url: str = "sqlite:///./app.db"

    class Config:
        env_file = ".env"
        env_file_encoding = "utf-8"

settings = Settings()
print("Settings from env/.env:", settings.app_name)

## Custom Validators

Use `@field_validator` for custom validation logic.

In [ ]:
from pydantic import field_validator

class Signup(BaseModel):
    email: str
    password: str

    @field_validator("email")
    @classmethod
    def email_must_contain_at(cls, v):
        if "@" not in v:
            raise ValueError("Invalid email")
        return v.lower()

    @field_validator("password")
    @classmethod
    def password_min_length(cls, v):
        if len(v) < 8:
            raise ValueError("Password must be 8+ chars")
        return v

s = Signup(email="A@B.com", password="longenough")
print(s.email)  # a@b.com (lowercased)

## Common Mistakes

- **Using model_dump() in older Pydantic**: Use dict() in v1, model_dump() in v2
- **Mutable default**: Use default_factory for list/dict
- **Validator order**: Validators run in definition order
- **Settings without pydantic-settings**: BaseSettings is in pydantic_settings in v2
- **Exposing password in response**: Use separate UserOut model

## Exercises

In [ ]:
# Exercise 1: Create a model with Field(description='...') and check the OpenAPI schema.
# YOUR CODE HERE

In [ ]:
# Exercise 2: Add a field_validator that ensures 'age' is between 0 and 150.
# YOUR CODE HERE

In [ ]:
# Exercise 3: Create UserCreate (name, email) and UserResponse (id, name, email). Use in FastAPI route.
# YOUR CODE HERE

In [ ]:
# Exercise 4: Use model_dump(exclude={'password'}) to create a safe dict from a user model.
# YOUR CODE HERE

In [ ]:
# Exercise 5: Create Settings with api_key: str (no default). Load from env API_KEY.
# YOUR CODE HERE

In [ ]:
# Exercise 6: Add List[Tag] to a Post model. Create Post with tags=[{'name':'a'}, {'name':'b'}].
# YOUR CODE HERE

## Key Takeaways

- Pydantic: BaseModel, Field(), field_validator
- model_dump(), model_dump_json(), exclude, exclude_unset
- Separate models for create vs response
- BaseSettings for config from env

## 🔜 Next: Day 3 - FastAPI Dependencies

Tomorrow: Dependency injection, middleware, and CORS!